# `quick-start.ipynb`

- `.env` 에 `TAVILY_API_KEY` 추가 필요

## Deep Agent
기존 Langchain Agent 보다 아래 능력을 포함
1. 계획 세우기
1. 파일 시스템 Tools
1. 하위 에이전트 다루는 능력

```sh
uv pip install deepagents tavily-python ipykernel python-dotenv langchain-openai
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
from typing import Literal
from tavily import TavilyClient
from langchain.tools import tool


TAVILY_API_KEY = os.getenv('TAVILY_API_KEY ')
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def internet_search(
    query: str,
    max_results: int = 5,  # = 뒤에는 기본값
    topic: Literal['general', 'news', 'finance'] = 'general',
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic
    )


In [ ]:
from deepagents import create_deep_agent

SYSTEM_PROMPT="""You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

deep_agent = create_deep_agent(
    model='openai:gpt-5-mini',
    tools=[internet_search],
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
question = """랭체인 1.0 버전 전과 후를 비교해줘. 
langchain, langgraph, langsmith 에 대해서 설명 및 달라진 점 말해주고,
deepagent가 새로 나왔는데 이것도 알려줘"""

result = deep_agent.invoke({
    'messages': [
        {'role': 'user', 'content': question}
    ]
})

In [ ]:
for msg in result['messages']:
    msg.pretty_print()